# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [15]:
class matrix:
    def __init__(self, n_or_values, m=None):
        if isinstance(n_or_values, int):
            if m is None:
                raise ValueError("You must provide both n and m.")

            self.data = []

            for i in range(n_or_values):
                row = []
                for j in range(m):
                    row.append(0.0)
                self.data.append(row)

        elif isinstance(n_or_values, list):
            if len(n_or_values) == 0:
                self.data = []
            else:
                number_of_columns = len(n_or_values[0])

                for row in n_or_values:
                    if not isinstance(row, list):
                        raise ValueError("Each row must be a list.")

                    if len(row) != number_of_columns:
                        raise ValueError("All rows must have the same length.")

                self.data = [row.copy() for row in n_or_values]

        else:
            raise TypeError("Use dimensions or a list of lists.")

    def __repr__(self):
        return str(self.data)

    def __getitem__(self, position):
        if isinstance(position, tuple):
            i, j = position
            return self.data[i][j]

        return self.data[position]

    def __setitem__(self, position, value):
        if isinstance(position, tuple):
            i, j = position
            self.data[i][j] = value
        else:
            self.data[position] = value

    def assign(self, other):
        if isinstance(other, matrix):
            new_values = other.data
        else:
            new_values = other

        if len(new_values) != len(self.data):
            raise ValueError("Matrices must have the same size.")

        if len(self.data) > 0:
            if len(new_values[0]) != len(self.data[0]):
                raise ValueError("Matrices must have the same size.")

        self.data = [row.copy() for row in new_values]

    def shape(self):
        if len(self.data) == 0:
            return (0, 0)

        return (len(self.data), len(self.data[0]))

    def transpose(self):
        rows, columns = self.shape()
        result = []

        for j in range(columns):
            new_row = []

            for i in range(rows):
                new_row.append(self.data[i][j])

            result.append(new_row)

        return matrix(result)

    def row(self, n):
        return matrix([self.data[n].copy()])

    def column(self, m):
        result = []

        for row in self.data:
            result.append([row[m]])

        return matrix(result)

    def to_list(self):
        return [row.copy() for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        result = []

        for row in self.data[n_0:n_1]:
            result.append(row[m_0:m_1])

        return matrix(result)

    def __getitem__(self, position):
        if isinstance(position, tuple):
            row_position, column_position = position

            if isinstance(row_position, slice):
                selected_rows = self.data[row_position]
            else:
                selected_rows = [self.data[row_position]]

            if isinstance(row_position, slice) or isinstance(column_position, slice):
                result = []

                for row in selected_rows:
                    if isinstance(column_position, slice):
                        result.append(row[column_position])
                    else:
                        result.append([row[column_position]])

                return matrix(result)

            return self.data[row_position][column_position]

        if isinstance(position, slice):
            return matrix(self.data[position])

        return self.data[position]

    def scalarmul(self, number):
        result = []

        for row in self.data:
            new_row = []

            for value in row:
                new_row.append(number * value)

            result.append(new_row)

        return matrix(result)

    def add(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same size.")

        result = []

        for i in range(self.shape()[0]):
            new_row = []

            for j in range(self.shape()[1]):
                new_row.append(self[i, j] + other[i, j])

            result.append(new_row)

        return matrix(result)

    def sub(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same size.")

        result = []

        for i in range(self.shape()[0]):
            new_row = []

            for j in range(self.shape()[1]):
                new_row.append(self[i, j] - other[i, j])

            result.append(new_row)

        return matrix(result)

    def mat_mult(self, other):
        if self.shape()[1] != other.shape()[0]:
            raise ValueError("Matrix sizes do not allow multiplication.")

        result = []

        for i in range(self.shape()[0]):
            new_row = []

            for j in range(other.shape()[1]):
                total = 0

                for k in range(self.shape()[1]):
                    total += self[i, k] * other[k, j]

                new_row.append(total)

            result.append(new_row)

        return matrix(result)

    def element_mult(self, other):
        if self.shape() != other.shape():
            raise ValueError("Matrices must have the same size.")

        result = []

        for i in range(self.shape()[0]):
            new_row = []

            for j in range(self.shape()[1]):
                new_row.append(self[i, j] * other[i, j])

            result.append(new_row)

        return matrix(result)

    def equals(self, other):
        if not isinstance(other, matrix):
            return False

        if self.shape() != other.shape():
            return False

        for i in range(self.shape()[0]):
            for j in range(self.shape()[1]):
                if self[i, j] != other[i, j]:
                    return False

        return True

    def __mul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        if isinstance(other, matrix):
            return self.mat_mult(other)

        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, (int, float)):
            return self.scalarmul(other)

        return NotImplemented

    def __add__(self, other):
        return self.add(other)

    def __sub__(self, other):
        return self.sub(other)

    def __eq__(self, other):
        return self.equals(other)

In [2]:
M1 = matrix(2, 3)
M2 = matrix([[1, 2, 3], [4, 5, 6]])

print("Zero matrix:", M1)
print("M2:", M2)

print("Using M2[0][1]:", M2[0][1])
print("Using M2[1, 2]:", M2[1, 2])

M1.assign(M2)
print("M1 after copying M2:", M1)

M1[0, 0] = 10
print("M1 after changing one value:", M1)

Zero matrix: [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
M2: [[1, 2, 3], [4, 5, 6]]
Using M2[0][1]: 2
Using M2[1, 2]: 6
M1 after copying M2: [[1, 2, 3], [4, 5, 6]]
M1 after changing one value: [[10, 2, 3], [4, 5, 6]]


In [8]:
M = matrix([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])

print("Shape:", M.shape())
print("Transpose:", M.transpose())
print("Row 1:", M.row(1))
print("Column 2:", M.column(2))
print("List copy:", M.to_list())
print("Block:", M.block(0, 2, 1, 3))
print("Slice:", M[0:2, 1:3])

Shape: (3, 3)
Transpose: [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
Row 1: [[4, 5, 6]]
Column 2: [[3], [6], [9]]
List copy: [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
Block: [[2, 3], [5, 6]]
Slice: [[2, 3], [5, 6]]


In [9]:
def constant(n, m, value):
    result = []

    for i in range(n):
        row = []

        for j in range(m):
            row.append(value)

        result.append(row)

    return matrix(result)


def zeros(n, m):
    return constant(n, m, 0.0)


def ones(n, m):
    return constant(n, m, 1.0)


def eye(n):
    result = []

    for i in range(n):
        row = []

        for j in range(n):
            if i == j:
                row.append(1.0)
            else:
                row.append(0.0)

        result.append(row)

    return matrix(result)

In [10]:
print("Constant matrix:", constant(2, 3, 5))
print("Zeros:", zeros(2, 3))
print("Ones:", ones(2, 3))
print("Identity matrix:", eye(3))

Constant matrix: [[5, 5, 5], [5, 5, 5]]
Zeros: [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
Ones: [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]]
Identity matrix: [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


In [13]:
A = matrix([
    [1, 2],
    [3, 4]
])

B = matrix([
    [5, 6],
    [7, 8]
])

print("Scalar multiplication:", A.scalarmul(2))
print("Addition:", A.add(B))
print("Subtraction:", A.sub(B))
print("Matrix multiplication:", A.mat_mult(B))
print("Element multiplication:", A.element_mult(B))
print("A equals A:", A.equals(A))
print("A equals B:", A.equals(B))

Scalar multiplication: [[2, 4], [6, 8]]
Addition: [[6, 8], [10, 12]]
Subtraction: [[-4, -4], [-4, -4]]
Matrix multiplication: [[19, 22], [43, 50]]
Element multiplication: [[5, 12], [21, 32]]
A equals A: True
A equals B: False


In [16]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[5, 6], [7, 8]])

print("2 * A:", 2 * A)
print("A * 2:", A * 2)
print("A + B:", A + B)
print("A - B:", A - B)
print("A * B:", A * B)
print("A == A:", A == A)
print("A == B:", A == B)

2 * A: [[2, 4], [6, 8]]
A * 2: [[2, 4], [6, 8]]
A + B: [[6, 8], [10, 12]]
A - B: [[-4, -4], [-4, -4]]
A * B: [[19, 22], [43, 50]]
A == A: True
A == B: False


In [17]:
A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [1, 0]])
C = matrix([[2, 0], [1, 2]])
I = eye(2)

left_1 = (A * B) * C
right_1 = A * (B * C)
print("(AB)C = A(BC):", left_1 == right_1)

left_2 = A * (B + C)
right_2 = (A * B) + (A * C)
print("A(B + C) = AB + AC:", left_2 == right_2)

print("AB:", A * B)
print("BA:", B * A)
print("AB != BA:", (A * B) != (B * A))

print("AI = A:", (A * I) == A)

(AB)C = A(BC): True
A(B + C) = AB + AC: True
AB: [[2, 1], [4, 3]]
BA: [[3, 4], [1, 2]]
AB != BA: True
AI = A: True
